# DCC 2026 Mission 3 — 환자 증상 인식

Google Colab에서 전사 텍스트만으로 9개 증상을 예측하는 TF-IDF + Logistic Regression 기준선을 학습하고 검증합니다. Training과 Validation은 엄격히 분리하며, Validation은 vocabulary/IDF 학습에 사용하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 경로 설정

`DATA_ROOT`에는 `001`~`012` part 폴더가 있어야 합니다. `CODE_DIR`에는 저장소의 `src/mission3` 폴더 안 세 파일을 업로드하세요. 아래 셀의 경로만 Drive 구조에 맞게 수정하면 됩니다.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/data(inho)')
CODE_DIR = Path('/content/drive/MyDrive/mission3')
MODEL_DIR = Path('/content/drive/MyDrive/saved_models/mission3')
MODEL_PATH = MODEL_DIR / 'mission3_tfidf_lr.joblib'
OUTPUT_DIR = Path('/content/drive/MyDrive/outputs')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'DATA_ROOT를 찾지 못했습니다: {DATA_ROOT}')
print('DATA_ROOT:', DATA_ROOT)
print('CODE_DIR :', CODE_DIR)

## 코드와 패키지 준비

Drive의 `mission3` 폴더에 [`common.py`](../src/mission3/common.py), [`train_tfidf_lr.py`](../src/mission3/train_tfidf_lr.py), [`inference.py`](../src/mission3/inference.py)를 넣은 뒤 실행합니다.

In [ ]:
!pip -q install scikit-learn joblib tqdm scipy

required_files = ['common.py', 'train_tfidf_lr.py', 'inference.py']
missing = [name for name in required_files if not (CODE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'CODE_DIR에 업로드되지 않은 파일: {missing}')
print('Mission 3 코드 준비 완료')

## 데이터 점검

9개 외 증상은 라벨에서만 제외되고 sample은 그대로 학습에 남아야 합니다. 이 셀은 서울 Training/Validation JSON 수와 target symptom 분포를 확인합니다.

In [ ]:
import json
from collections import Counter

TARGET_SYMPTOMS = ('고열', '구토', '두통', '복통', '어지러움', '열상', '오심', '전신쇠약', '호흡곤란')

def json_paths(split):
    label_folder = 'TL_서울_구급' if split == 'Training' else 'VL_서울_구급'
    paths = []
    for part in sorted(path for path in DATA_ROOT.iterdir() if path.is_dir()):
        label_dir = part / '대학부 데이터' / split / '2.라벨링데이터' / label_folder
        if label_dir.is_dir():
            paths.extend(label_dir.rglob('*.json'))
    return sorted(paths)

train_jsons, val_jsons = json_paths('Training'), json_paths('Validation')
if not train_jsons or not val_jsons:
    raise FileNotFoundError('서울 Training/Validation 라벨 폴더를 모두 찾지 못했습니다.')

# Drive에서 수만 개 파일을 전부 열면 Colab에서 매우 오래 걸릴 수 있습니다.
# 이 셀은 빠른 점검용으로 최대 500개만 확인합니다. 전체 JSON은 다음 학습 셀에서 사용됩니다.
CHECK_SAMPLE_SIZE = 500
check_jsons = train_jsons[:CHECK_SAMPLE_SIZE]
counts = Counter()
zero_target_count = 0
for path in check_jsons:
    with path.open(encoding='utf-8') as file:
        annotation = json.load(file)
    raw = annotation.get('symptom', [])
    raw = [raw] if isinstance(raw, str) else raw
    labels = {item.strip() for item in raw if isinstance(item, str)}
    targets = labels.intersection(TARGET_SYMPTOMS)
    counts.update(targets)
    zero_target_count += not bool(targets)

print(f'Training JSON 전체: {len(train_jsons):,}')
print(f'Validation JSON: {len(val_jsons):,}')
print(f'점검에 사용한 Training JSON: {len(check_jsons):,}')
print(f'점검 sample 중 target 0개: {zero_target_count:,}')
for symptom in TARGET_SYMPTOMS:
    print(f'{symptom:6s}: {counts[symptom]:,}')

## 학습 및 Validation 평가

기본 설정은 word 1–2 gram + character 2–5 gram TF-IDF입니다. 모델은 Training만으로 fit한 뒤 Validation의 클래스별 F1과 Macro F1을 출력하고 checkpoint에 저장합니다.

In [ ]:
%cd {CODE_DIR}
# Colab CPU용 빠른 baseline 설정입니다 (word 4만 + char 4만 feature).
!python train_tfidf_lr.py --data_root "{DATA_ROOT}" --ckpt_path "{MODEL_PATH}" --min_df 3 --max_features 40000 --C 2.0

if not MODEL_PATH.is_file():
    raise RuntimeError(f'학습 checkpoint가 생성되지 않았습니다: {MODEL_PATH}')

In [ ]:
import joblib

checkpoint = joblib.load(MODEL_PATH)
print(f"Validation Macro F1: {checkpoint['validation_macro_f1']:.4f}")
for symptom, threshold, score in zip(
    checkpoint['target_symptoms'], checkpoint['thresholds'], checkpoint['validation_class_f1']
):
    print(f'{symptom:6s} threshold={threshold:.2f}, F1={score:.4f}')

## 제출 CSV 생성

평가 JSON 폴더를 `EVAL_LABEL_DIR`에 지정해 실행합니다. Mission 3는 전사 text만 사용하므로 `EVAL_AUDIO_DIR`는 공통 실행 규격을 위한 인자이며 읽지 않습니다.

In [ ]:
# 실제 평가 데이터 폴더로 바꾼 뒤 실행하세요.
EVAL_LABEL_DIR = Path('/content/drive/MyDrive/evaluation/2.라벨링데이터')
EVAL_AUDIO_DIR = Path('/content/drive/MyDrive/evaluation/1.원천데이터')
SUBMISSION_PATH = OUTPUT_DIR / 'mission3.csv'

if not EVAL_LABEL_DIR.is_dir():
    raise FileNotFoundError(f'EVAL_LABEL_DIR를 실제 평가 JSON 폴더로 수정하세요: {EVAL_LABEL_DIR}')
%cd {CODE_DIR}
!python inference.py --audio_dir "{EVAL_AUDIO_DIR}" --label_dir "{EVAL_LABEL_DIR}" --ckpt_path "{MODEL_PATH}" --output "{SUBMISSION_PATH}"

import pandas as pd
submission = pd.read_csv(SUBMISSION_PATH)
display(submission.head())
print('저장 위치:', SUBMISSION_PATH)
print('행 수:', len(submission))